<h1> Feature Extraction using chatGPT and the OpenAI Python API library

In [1]:
import openai
from openai import OpenAI
import pandas as pd

In [2]:
# Prepare the connection through the OpenAI Python API library. Credentials will be read from the environment file by default.
client = OpenAI()

In [4]:
SAMPLE_SIZE = 100
FILE_PATH = f"../data/spotify_reviews_post-2023_{SAMPLE_SIZE}.json"

# Fetch relevant data
df = pd.read_csv('../data/spotify_reviews.csv')

# Filter and format relevant data
df = df.drop(columns=['reviewId', 'userName', 'score', 'thumbsUpCount', 'reviewCreatedVersion'])
df['at'] = pd.to_datetime(df['at'])
df = df[df['at'] > '2023-01-01']
df.sample(SAMPLE_SIZE)
 
# Save data as JSON file
df.to_json(FILE_PATH, orient='records')

In [7]:
# Create the Assistant
print("Creating assistant...")
assistant = client.beta.assistants.create(
    name="Spotify Review Analyzer",
    instructions="You are a helpful assistant analyzing Spotify app reviews. Be precise and focus on specific features which are highlighted in the reviews.",
    model="gpt-3.5-turbo",
    tools=[{"type": "file_search"}],
    temperature=0.5
)

# Create a vector store to store the data
print("Creating vector store...")
vector_store = client.beta.vector_stores.create(name="Spotify Review Vector Store ({SAMPLE_SIZE})")

# Prepare files for upload to OpenAI
file_paths = [FILE_PATH]
file_streams = [open(path, "rb") for path in file_paths]

# Use SDK helper to upload the files, add them to the vector store and poll the status of the file batch for completion.
print("Uploading files...")
file_batch = client.beta.vector_stores.file_batches.upload_and_poll(
  vector_store_id=vector_store.id, files=file_streams
)

# You can print the status and the file counts of the batch to see the result of this operation.
print("Finished uploading files.")
print(file_batch.status)
print(file_batch.file_counts)

# Update the assistant with the vector store
assistant = client.beta.assistants.update(
  assistant_id=assistant.id,
  tool_resources={"file_search": {"vector_store_ids": [vector_store.id]}},
)

completed
FileCounts(cancelled=0, completed=1, failed=0, in_progress=0, total=1)


In [ ]:
# Create the Thread
thread = openai.beta.threads.create()

# Add individual review to the thread
def add_user_message(message):
    openai.beta.threads.messages.create(
        thread_id=thread.id,
        role="user",
        content=message
    )

# Function to run the assistant on the thread
def run_assistant():
    run = openai.beta.threads.runs.create(
        thread_id=thread.id,
        assistant_id=assistant.id
    )
    return run

# Function to retrieve and process assistant's response
def get_assistant_response(run_id):
    messages = openai.beta.threads.messages.list(
        thread_id=thread.id
    )
    response_message = messages.data[0]
    return response_message.content[0].text.value

message = "Please extract the specific features that users dislike about the Spotify app based on the reviews data."
add_user_message(message)
run = run_assistant()
while run.status != "completed":
    run = openai.beta.threads.runs.retrieve(thread_id=thread.id, run_id=run.id)
assistant_response = get_assistant_response(run.id)
print(assistant_response)

Several users have expressed dissatisfaction with the Spotify app due to various issues, including the requirement to have a premium subscription for basic features like turning off shuffle, peeking into the next song, or making a queue. Users have also reported an increase in ads, glitches, and bugs, as well as feeling pressured to buy premium despite the app's instability. One user specifically mentioned that the "30 minutes" of ad-free music promised by Spotify only lasted about 6 minutes, with the company attributing this discrepancy to an "error" being "worked on"【4:0†source】【4:1†source】.
Users have expressed frustration with Spotify, mentioning issues such as the app becoming worse over time, the heavy emphasis on premium features, and the abundance of ads. Some users have recommended switching to other music apps like Wynk Music due to their dissatisfaction with Spotify's performance and features【8:0†source】【8:3†source】.
